승객 생존율 예측하기
  - 다양한 머신러닝 모델 적용 및 머신러닝 기법에 적용하여 각  ML모델의 특성과 사용방법을 정리 및 요약하기

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
#8가지 분류 모델 라이브러리 가져오기
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import Perceptron, SGDClassifier

In [2]:
# 데이터 불러오기
url = "https://raw.githubusercontent.com/sehakflower/data/main/titanic.csv"
df = pd.read_csv(url, sep='\t')
display(df.head(156))

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
151,152,1,1,"Pears, Mrs. Thomas (Edith Wearne)",female,22.0,1,0,113776,66.6000,C2,S
152,153,0,3,"Meo, Mr. Alfonzo",male,55.5,0,0,A.5. 11206,8.0500,NaN,S
153,154,0,3,"van Billiard, Mr. Austin Blyler",male,40.5,0,2,A/5. 851,14.5000,NaN,S
154,155,0,3,"Olsen, Mr. Ole Martin",male,NaN,0,0,Fa 265302,7.3125,NaN,S


In [3]:
#데이터 타입 출력해보기
display(df.dtypes)

,0
PassengerId,int64
Survived,int64
Pclass,int64
Name,object
Sex,object
Age,float64
SibSp,int64
Parch,int64
Ticket,object
Fare,float64


In [4]:
#1. name 열을 5개로 구분하고 숫자형 데이터로 바꾸기 (호칭 추출)
# 이름에서 Mr, Mrs, Miss, Master 등 찾아내기

df['Title'] = df['Name'].str.extract(r'([A-Za-z]+)\.', expand = False)

# 자주 안나오는 호칭은 'Rare'로 통일해서 총 5개 군으로 맞춤
rare_titles = ['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
df['Title'] = df['Title'].replace(rare_titles, 'Rare')
df['Title'] = df['Title'].replace({'MIIE': 'Miss.', 'Mme':'Mrs.' ,'Ms' : 'Miss.'})

#글자로 된 호칭을 머신러닝 모델이 계산할 수 있게 숫자로 매핑함
title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
df['Title'] = df['Title'].map(title_mapping).fillna(0).astype(int)
df['Title']


,Title
0,1
1,3
2,2
3,3
4,1
...,...
151,3
152,1
153,1
154,1


In [5]:
# 'Title' 열의 호칭별 데이터 개수 확인
display(df['Title'].value_counts())

,count
Title,
1,89
2,34
3,22
4,8
5,3


In [6]:
# age 열의 빈 값을 호칭별 중앙값(Median)으로 바꾸기
# 나이가 비어 있는 사람은 그 사람의 호칭의 중앙값으로 채우기
df['Age'] = df['Age'].fillna(df.groupby('Title')['Age'].transform('median'))

In [7]:
#  male : 0 , female : 1 로 바꾸기
df['Sex'] = df['Sex'].map({'male' : 0, 'female' : 1})
df['Sex']

,Sex
0,0
1,1
2,1
3,1
4,0
...,...
151,1
152,0
153,0
154,0


In [8]:
# embarked 열 숫자로 바꾸기(빈 값은 가장 많은 's' 로 채우고 변환하기)
embarked_mapping = {'S': 1, 'C': 2, 'Q': 3}
df['Embarked'] = df['Embarked'].map(embarked_mapping)


df['Embarked']

,Embarked
0,1.0
1,2.0
2,1.0
3,1.0
4,1.0
...,...
151,1.0
152,1.0
153,1.0
154,1.0


In [9]:
# fare열, 개인별 요금을 나타내는 fare_person  열 추가
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['Fare_Person'] = df['Fare'] / df['FamilySize']

df['FamilySize']
df['Fare_Person']

,Fare_Person
0,3.625000
1,35.641650
2,7.925000
3,26.550000
4,8.050000
...,...
151,33.300000
152,8.050000
153,4.833333
154,7.312500


In [10]:
#  혹시 모를 요금의 빈 값(결측치) 처리
df['Fare'] = df['Fare'].fillna(df['Fare'].median())
df['Fare_Person'] = df['Fare_Person'].fillna(df['Fare_Person'].median())


In [11]:
# 최종 데이터 피처(x)와 타겟(y, 생존여부) 선택
features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Title', 'FamilySize', 'Fare_Person']
x = df[features]
y = df['Survived']

In [12]:
# 8개 분류 모델 학습 및 테스트 정확도 비교

In [13]:
# 전체 데이터를 공부용(80%)과 실제 시험용(20%) 데이터로 나누기.
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# 거리나 기울기를 계산하는 모델들을 위해 데이터 단위(Scale)를 통일해 줍니다.
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [14]:
models = {
     "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "Logistic Regression": LogisticRegression(C=1.0, penalty='l2', solver='lbfgs', random_state=42),
    "Gaussian Naive Bayes": GaussianNB(),
    "K-Nearest Neighbor": KNeighborsClassifier(n_neighbors=5),
    "Linear SVM": LinearSVC(C=1.0, random_state=42, max_iter=10000),
    "Perceptron": Perceptron(max_iter=1000, eta0=1.0, random_state=42),
    "SGD Classifier": SGDClassifier(loss='log_loss', penalty='l2', random_state=42)

}
accuracy_results = {}

In [18]:
for name, model in models.items():
    # 나무 계열 모델은 가공하지 않은 원본 데이터를, 그 외 모델은 크기를 맞춘 데이터를 사용
    if name in ["Decision Tree", "Random Forest"]:
        model.fit(x_train, y_train)
        score = model.score(x_test, y_test)
    else:
        # 스케일링된 데이터에 NaN이 있을 경우 임시로 0으로 대체
        current_X_train_scaled = np.nan_to_num(x_train_scaled)
        current_X_test_scaled = np.nan_to_num(x_test_scaled)
        model.fit(current_X_train_scaled, y_train)
        score = model.score(current_X_test_scaled, y_test)

    accuracy_results[name] = score

    print(score)

0.75
0.71875
0.65625
0.6875
0.625
0.65625
0.6875
0.6875


In [24]:
# 모델의 정확도 비교분석하기

# 각 모델을 하나씩 학습시키고 정확도를 측정합니다.
for name, model in models.items():
    # 나무 계열 모델은 가공하지 않은 원본 데이터를, 그 외 모델은 크기를 맞춘 데이터를 사용합니다.
    if name in ["Decision Tree", "Random Forest"]:
        model.fit(x_train, y_train)
        score = model.score(x_test, y_test)
    else:
        # 스케일링된 데이터에 NaN이 있을 경우 임시로 0으로 대체
        current_X_train_scaled = np.nan_to_num(x_train_scaled)
        current_X_test_scaled = np.nan_to_num(x_test_scaled)
        model.fit(current_X_train_scaled, y_train)
        score = model.score(current_X_test_scaled, y_test)

    accuracy_results[name] = score

    print(score)

# ==========================================
# [Step 5] 랜덤 포레스트 모델 교차 검증 (Cross Validation)
# ==========================================
# 5번 데이터를 나누어 평균 성능을 평가하는 5-Fold 교차 검증을 진행합니다.
rf_model = models["Random Forest"]
cv_scores = cross_val_score(rf_model, x, y, cv=5)

# ==========================================
# [Step 6] 결과 출력하기
# ==========================================
print("## [결과 1] 8개 머신러닝 모델의 정확도 순위")
print("-" * 55)
sorted_rank = sorted(accuracy_results.items(), key=lambda x: x[1], reverse=True)
for index, (model_name, acc) in enumerate(sorted_rank, 1):
    print(f"{index:2d}위. {model_name:<25} : {acc * 100:.2f}%")

print("\n" + "="*55 + "\n")

best_model_name, best_model_acc = sorted_rank[0]
print(f"🏆 가장 높은 정확도를 보여주는 모델: {best_model_name} ({best_model_acc * 100:.2f}%)")

print("\n" + "="*55 + "\n")

print("## [결과 2] 랜덤 포레스트 모델 5-Fold 교차 검증")
print("-" * 55)
print(f"각 회차별 시험 점수: {cv_scores}")
print(f"교차 검증 최종 평균 점수: {cv_scores.mean() * 100:.2f}%")

0.75
0.71875
0.65625
0.6875
0.625
0.65625
0.6875
0.6875
## [결과 1] 8개 머신러닝 모델의 정확도 순위
-------------------------------------------------------
 1위. Decision Tree             : 75.00%
 2위. Random Forest             : 71.88%
 3위. Gaussian Naive Bayes      : 68.75%
 4위. Perceptron                : 68.75%
 5위. SGD Classifier            : 68.75%
 6위. Logistic Regression       : 65.62%
 7위. Linear SVM                : 65.62%
 8위. K-Nearest Neighbor        : 62.50%


🏆 가장 높은 정확도를 보여주는 모델: Decision Tree (75.00%)


## [결과 2] 랜덤 포레스트 모델 5-Fold 교차 검증
-------------------------------------------------------
각 회차별 시험 점수: [0.75       0.83870968 0.83870968 0.77419355 0.74193548]
교차 검증 최종 평균 점수: 78.87%
